# Lab | LangChain Med

## Objectives

- continue on with lesson 2' example, use different datasets to test what we did in class. Some datasets are suggested in the notebook but feel free to scout other datasets on HuggingFace or Kaggle.
- Find another model on Hugging Face and compare it.
- Modify the prompt to fit your selected dataset.

In [1]:
# Basic data libraries.
# pandas = table/dataframe work.
# numpy = numerical operations.
import numpy as np
import pandas as pd


## Load the Dataset
As you can see the notebook is ready to work with three different Datasets. Just uncomment the lines of the Dataset you want to use.

I selected Datasets with News. Two of them have just a brief decription of the news, but the other contains the full text.

As we are working in a free and limited space, I limited the number of news to use with the variable MAX_NEWS. Feel free to pull more if you have memory available.

The name of the field containing the text of the new is stored in the variable *DOCUMENT* and the metadata in *TOPIC*

In [2]:
# ------------------------------------------------------------
# Dataset setup
# ------------------------------------------------------------
# This notebook is written so you can choose ONE dataset.
# The important idea is: choose the text column and metadata column.
#
# DOCUMENT = the column that contains the text we want to store/retrieve.
# TOPIC = a column used as metadata, like category/title/topic.
#
# In a LangChain version, this step is similar to using:
#   CSVLoader / DataFrameLoader / JSONLoader
# and then choosing which column becomes page_content and metadata.

# Example 1: labelled news dataset
# news = pd.read_csv('/content/labelled_newscatcher_dataset.csv', sep=';')
# MAX_NEWS = 1000
# DOCUMENT = "title"
# TOPIC = "topic"

# Example 2: BBC news dataset
# news = pd.read_csv('/kaggle/input/bbc-news/bbc_news.csv')
# MAX_NEWS = 1000
# DOCUMENT = "description"
# TOPIC = "title"

# Example 3: MIT AI news dataset
news = pd.read_csv('/Users/omari/Documents/Academic/Ironhack_AI_Bootcamp/Bootcamp_W7/D5/lab-langchain-med/datasets/articles.csv')
MAX_NEWS = 100
DOCUMENT = "Article Body"
TOPIC = "Article Header"

# Choose one dataset above and uncomment it.
#news = "PICK A DATASET"


ChromaDB requires that the data has a unique identifier. We can make it with this statement, which will create a new column called **Id**.


In [3]:
# Quick check: display the loaded dataframe.
# This helps confirm the dataset loaded correctly and the columns exist.
# LangChain equivalent: inspect documents after loading with loader.load().
news


,Unnamed: 0,Published Date,Author,Source,Article Header,Sub_Headings,Article Body,Url
0,0,"July 7, 2023",Adam Zewe,MIT News Office,Learning the language of molecules to predict ...,This AI system only needs a small amount of da...,['Discovering new materials and drugs typicall...,https://news.mit.edu/2023/learning-language-mo...
1,1,"July 6, 2023",Alex Ouyang,Abdul Latif Jameel Clinic for Machine Learning...,MIT scientists build a system that can generat...,"BioAutoMATED, an open-source, automated machin...",['Is it possible to build machine-learning mod...,https://news.mit.edu/2023/bioautomated-open-so...
2,2,"June 30, 2023",Jennifer Michalowski,McGovern Institute for Brain Research,"When computer vision works more like a brain, ...",Training artificial neural networks with data ...,"['From cameras to self-driving cars, many of t...",https://news.mit.edu/2023/when-computer-vision...
3,3,"June 30, 2023",Mary Beth Gallagher,School of Engineering,Educating national security leaders on artific...,"Experts from MIT’s School of Engineering, Schw...",['Understanding artificial intelligence and ho...,https://news.mit.edu/2023/educating-national-s...
4,4,"June 30, 2023",Adam Zewe,MIT News Office,Researchers teach an AI to write better chart ...,A new dataset can help scientists develop auto...,['Chart captions that explain complex trends a...,https://news.mit.edu/2023/researchers-chart-ca...
...,...,...,...,...,...,...,...,...
1013,1013,"September 13, 1994",NaN,NaN,MIT's Robotic Fish Takes First Swim,NaN,"[""CAMBRIDGE, Mass.--Consider the fish: highly ...",https://news.mit.edu/1994/robotuna
1014,1014,"June 29, 1994","Donna Coveney, News Office",NaN,Can a robot teach us how people learn?,NaN,"['Cog, the newest and most ambitious robot dev...",https://news.mit.edu/1994/robot-0629
1015,1015,"May 18, 1994","Alice C. Waugh, News Office",NaN,Micro-robot holds promise for new surgical tec...,NaN,"['A few years from now, patients with polyps o...",https://news.mit.edu/1994/microrobot-0518
1016,1016,"April 13, 1994","Andrea Cohen, MIT Sea Grant",NaN,Robot journeys beneath arctic ice,NaN,"[""Despite a wind chill index of -80oF and an e...",https://news.mit.edu/1994/robot-0413


In [4]:
# ChromaDB needs each stored document to have a unique ID.
# Here we create an ID from the dataframe index.
# LangChain can create IDs for you in some vector stores, but raw Chroma requires explicit IDs.
news["id"] = news.index
news.head()


,Unnamed: 0,Published Date,Author,Source,Article Header,Sub_Headings,Article Body,Url,id
0,0,"July 7, 2023",Adam Zewe,MIT News Office,Learning the language of molecules to predict ...,This AI system only needs a small amount of da...,['Discovering new materials and drugs typicall...,https://news.mit.edu/2023/learning-language-mo...,0
1,1,"July 6, 2023",Alex Ouyang,Abdul Latif Jameel Clinic for Machine Learning...,MIT scientists build a system that can generat...,"BioAutoMATED, an open-source, automated machin...",['Is it possible to build machine-learning mod...,https://news.mit.edu/2023/bioautomated-open-so...,1
2,2,"June 30, 2023",Jennifer Michalowski,McGovern Institute for Brain Research,"When computer vision works more like a brain, ...",Training artificial neural networks with data ...,"['From cameras to self-driving cars, many of t...",https://news.mit.edu/2023/when-computer-vision...,2
3,3,"June 30, 2023",Mary Beth Gallagher,School of Engineering,Educating national security leaders on artific...,"Experts from MIT’s School of Engineering, Schw...",['Understanding artificial intelligence and ho...,https://news.mit.edu/2023/educating-national-s...,3
4,4,"June 30, 2023",Adam Zewe,MIT News Office,Researchers teach an AI to write better chart ...,A new dataset can help scientists develop auto...,['Chart captions that explain complex trends a...,https://news.mit.edu/2023/researchers-chart-ca...,4


In [5]:
# Keep only a small subset so the lab runs faster.
# This is not conceptually required; it is just for speed/memory.
# LangChain equivalent: docs = docs[:MAX_NEWS]
subset_news = news.head(MAX_NEWS)


## Import and configure the Vector Database
I'm going to use ChromaDB, the most popular OpenSource embedding Database.

First we need to import ChromaDB, and after that import the **Settings** class from **chromadb.config** module. This class allows us to change the setting for the ChromaDB system, and customize its behavior.

In [6]:
# Install raw ChromaDB.
# This is NOT the LangChain wrapper.
# Raw ChromaDB gives you direct access to client, collection, add, query.
!pip install chromadb


In [7]:
# Raw ChromaDB imports.
# chromadb.Client / PersistentClient = direct database connection.
# Settings can configure storage behavior in older/newer Chroma setups.
#
# LangChain equivalent import would usually be:
#   from langchain_chroma import Chroma
import chromadb
from chromadb.config import Settings


Now we need to create the seetings object calling the **Settings** function imported previously. We store the object in the variable **settings_chroma**.

Is necessary to inform two parameters
* chroma_db_impl. Here we specify the database implementation and the format how store the data. I choose ***duckdb***, because his high-performace. It operate primarly in memory. And is fully compatible with SQL. The store format ***parquet*** is good for tabular data. With good compression rates and performance.

* persist_directory: It just contains the directory where the data will be stored. Is possible work without a directory and the data will be stored in memory without persistece, but Kaggle dosn't support that.

In [8]:
# Create a persistent Chroma client.
# Persistent means vectors are saved on disk, not only in memory.
#
# Raw ChromaDB style:
#   chroma_client = chromadb.PersistentClient(path="...")
#
# LangChain equivalent:
#   vectorstore = Chroma.from_documents(..., persist_directory="...")
#
# Note: replace this placeholder path with a real folder if you run it locally.
chroma_client = chromadb.PersistentClient(path="/Users/omari/Documents/Academic/Ironhack_AI_Bootcamp/Bootcamp_W7/D5/lab-langchain-med")


## Filling and Querying the ChromaDB Database
The Data in ChromaDB is stored in collections. If the collection exist we need to delete it.

In the next lines, we are creating the collection by calling the ***create_collection*** function in the ***chroma_client*** created above.

In [9]:
# A Chroma collection is like a table/index of documents and embeddings.
# If the collection already exists, this notebook deletes it so the run starts fresh.
#
# Raw ChromaDB style:
#   create_collection()
#
# LangChain equivalent:
#   Chroma.from_documents(..., collection_name="news_collection")
collection_name = "news_collection"
if len(chroma_client.list_collections()) > 0 and collection_name in [chroma_client.list_collections()[0].name]:
        chroma_client.delete_collection(name=collection_name)

collection = chroma_client.create_collection(name=collection_name)


It's time to add the data to the collection. Using the function ***add*** we need to inform, at least ***documents***, ***metadatas*** and ***ids***.
* In the **document** we store the big text, it's a different column in each Dataset.
* In **metadatas**, we can informa a list of topics.
* In **id** we need to inform an unique identificator for each row. It MUST be unique! I'm creating the ID using the range of MAX_NEWS.


In [10]:
# Add documents to raw ChromaDB.
#
# documents = the text that will be embedded and searched.
# metadatas = extra labels attached to each document.
# ids = unique identifiers required by ChromaDB.
#
# Raw ChromaDB automatically uses its default embedding function unless configured otherwise.
#
# LangChain equivalent:
#   docs = [Document(page_content=text, metadata={TOPIC: topic}) ...]
#   vectorstore = Chroma.from_documents(docs, embedding=embeddings)
collection.add(
    documents=subset_news[DOCUMENT].tolist(),
    metadatas=[{TOPIC: topic} for topic in subset_news[TOPIC].tolist()],
    ids=[f"id{x}" for x in range(MAX_NEWS)],
)


In [11]:
# Query raw ChromaDB directly.
# query_texts is the user/search question.
# n_results controls how many similar documents to return.
#
# Raw ChromaDB returns a dictionary with documents, ids, distances, metadata.
#
# LangChain equivalent:
#   retriever = vectorstore.as_retriever(search_kwargs={"k": 10})
#   results = retriever.invoke("laptop")
results = collection.query(query_texts=["laptop"], n_results=10)

print(results)


{'ids': [['id9', 'id85', 'id1', 'id80', 'id52', 'id27', 'id73', 'id93', 'id6', 'id98']], 'embeddings': None, 'documents': [['[\'How will advances in computing transform human society?\', \'MIT students contemplated this impending question as part of the Envisioning the Future of Computing Prize — an essay contest in which they were challenged to imagine ways that computing technologies could improve our lives, as well as the pitfalls and dangers associated with them.\', \'Offered for the first time this year, the Institute-wide competition invited MIT undergraduate and graduate students to share their ideas, aspirations, and vision for what they think a future propelled by advancements in computing holds. Nearly 60 students put pen to paper, including those majoring in mathematics, philosophy, electrical engineering and computer science, brain and cognitive sciences, chemical engineering, urban studies and planning, and management, and entered their submissions.\', \'Students dreamed u

## Vector MAP

In [12]:
#%pip install -q matplotlib scikit-learn


In [13]:
# Optional visualization tools.
# PCA reduces high-dimensional embeddings to 2D so we can plot them.
# This is only for visualization, not for retrieval itself.
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA


In [14]:
# Get one stored item from Chroma by ID.
# include=["documents", "embeddings"] asks Chroma to return both text and vector.
#
# This helps you see that every document becomes:
#   document text + embedding vector + metadata + id
#
# LangChain equivalent:
#   Usually you inspect retrieved Document objects, not raw vectors.
getado = collection.get(ids="id141",
                       include=["documents", "embeddings"])


In [15]:
# Separate the returned embedding vector and document text.
# word_vectors = numerical representation of the document.
# word_list = original text.
#
# This is useful for understanding that vector search works on embeddings,
# while the LLM later reads the original text.
word_vectors = getado["embeddings"]
word_list = getado["documents"]
# word_vectors


Once we have our information inside the Database we can query It, and ask for data that matches our needs. The search is done inside the content of the document, and it dosn't look for the exact word, or phrase. The results will be based on the similarity between the search terms and the content of documents.

The metadata is not used in the search, but they can be utilized for filtering or refining the results after the initial search.


## Loading the model and creating the prompt
TRANSFORMERS!!
Time to use the library **transformers**, the most famous library from [hugging face](https://huggingface.co/) for working with language models.

We are importing:
* **Autotokenizer**: It is a utility class for tokenizing text inputs that are compatible with various pre-trained language models.
* **AutoModelForCasualLLM**: it provides an interface to pre-trained language models specifically designed for language generation tasks using causal language modeling (e.g., GPT models), or the model used in this notebook ***databricks/dolly-v2-3b***.
* **pipeline**: provides a simple interface for performing various natural language processing (NLP) tasks, such as text generation (our case) or text classification.

The model selected is [dolly-v2-3b](https://huggingface.co/databricks/dolly-v2-3b), the smallest Dolly model. It have 3billion paramaters, more than enough for our sample, and works much better than GPT2.

Please, feel free to test [different Models](https://huggingface.co/models?pipeline_tag=text-generation&sort=trending), you need to search for NLP models trained for text-generation. My recomendation is choose "small" models, or we will run out of memory in kaggle.  


In [16]:
%pip install -q torch


Note: you may need to restart the kernel to use updated packages.


In [17]:
# ------------------------------------------------------------
# Load a Hugging Face text-generation model
# ------------------------------------------------------------
# This part is NOT LangChain. It uses transformers directly.
#
# LangChain equivalent would be something like:
#   llm = ChatOpenAI(...)
# or a HuggingFacePipeline wrapper.
#
# The notebook uses GPT-2 as a fallback because Dolly was difficult to load.
# GPT-2 is not instruction-tuned, so answers may be weak.
!pip install --upgrade transformers huggingface_hub

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

model_id = "gpt2"

# Tokenizer converts text into token IDs the model can read.
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

# Model generates text from token IDs.
lm_model = AutoModelForCausalLM.from_pretrained(model_id)


/opt/anaconda3/envs/ironhack_langchain/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 148/148 [00:00<00:00, 6938.24it/s]


The next step is to initialize the pipeline using the objects created above.

The model's response is limited to 256 tokens, for this project I'm not interested in a longer response, but it can easily be extended to whatever length you want.

Setting ***device_map*** to ***auto*** we are instructing the model to automaticaly select the most appropiate device: CPU or GPU for processing the text generation.  

In [18]:
# Create a Hugging Face generation pipeline.
# This wraps tokenization + model generation into one callable object.
#
# Raw transformers style:
#   lm_response = pipe(prompt)
#
# LangChain equivalent:
#   response = llm.invoke(prompt)
pipe = pipeline(
    "text-generation",
    model=lm_model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    device_map="auto",
)


[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


## Creating the extended prompt
To create the prompt we use the result from query the Vector Database  and the sentence introduced by the user.

The prompt have two parts, the **relevant context** that is the information recovered from the database and the **user's question**.

We only need to join the two parts together to create the prompt that we are going to send to the model.

You can limit the lenght of the context passed to the model, because we can get some Memory problems with one of the datasets that contains a realy large text in the document part.

In [21]:
# ------------------------------------------------------------
# Manual RAG prompt construction
# ------------------------------------------------------------
# We take documents retrieved from Chroma and join them into one context string.
# Then we combine:
#   relevant context + user question
#
# This is the manual version of a RAG prompt.
#
# LangChain equivalent:
#   prompt = ChatPromptTemplate.from_template("Context: {context} ... Question: {question}")
question = "Can I buy a Toshiba laptop?"
context = " ".join([f"#{str(i)}" for i in results["documents"][0]])

# If the context is too long for the model, slice it.
context = context[0:2000]

prompt_template = f"""
Relevant context:
{context}

Question:
{question}

Answer:
"""
prompt_template


"\nRelevant context:\n#['How will advances in computing transform human society?', 'MIT students contemplated this impending question as part of the Envisioning the Future of Computing Prize — an essay contest in which they were challenged to imagine ways that computing technologies could improve our lives, as well as the pitfalls and dangers associated with them.', 'Offered for the first time this year, the Institute-wide competition invited MIT undergraduate and graduate students to share their ideas, aspirations, and vision for what they think a future propelled by advancements in computing holds. Nearly 60 students put pen to paper, including those majoring in mathematics, philosophy, electrical engineering and computer science, brain and cognitive sciences, chemical engineering, urban studies and planning, and management, and entered their submissions.', 'Students dreamed up highly inventive scenarios for how the technologies of today and tomorrow could impact society, for better 

Now all that remains is to send the prompt to the model and wait for its response!


In [22]:
# Send the final prompt to the Hugging Face pipeline.
# The model sees both the retrieved context and the question.
#
# LangChain equivalent:
#   answer = chain.invoke(question)
# or:
#   answer = llm.invoke(prompt_template)
lm_response = pipe(prompt_template)
print(lm_response[0]["generated_text"])


[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



Relevant context:
#['How will advances in computing transform human society?', 'MIT students contemplated this impending question as part of the Envisioning the Future of Computing Prize — an essay contest in which they were challenged to imagine ways that computing technologies could improve our lives, as well as the pitfalls and dangers associated with them.', 'Offered for the first time this year, the Institute-wide competition invited MIT undergraduate and graduate students to share their ideas, aspirations, and vision for what they think a future propelled by advancements in computing holds. Nearly 60 students put pen to paper, including those majoring in mathematics, philosophy, electrical engineering and computer science, brain and cognitive sciences, chemical engineering, urban studies and planning, and management, and entered their submissions.', 'Students dreamed up highly inventive scenarios for how the technologies of today and tomorrow could impact society, for better or 